# BERT-Based News Categorization Engine
In this notebook, we develop an NLP pipeline to automatically classify news headlines. We leverage the AG News dataset to fine-tune a pre-trained **BERT** architecture, enabling it to accurately sort text into four distinct topics:
* Global News
* Sports Events
* Finance & Business
* Technology & Science

In [ ]:
from datasets import load_dataset
from transformers import BertTokenizer
import torch

### 1. Data Acquisition: AG News Dataset
We fetch our training and testing data directly via the Hugging Face `datasets` library. The data consists of short news texts grouped by four main labels.

In [ ]:
dataset = load_dataset("ag_news")
dataset

In [ ]:
dataset["train"][0]

In [ ]:
dataset["train"].features

### 2. Initializing the Tokenizer
**Text Tokenization Process**

Machine learning models cannot process raw text. We use the BERT tokenizer to translate our news headlines into numerical representations (input IDs and attention masks) that the transformer algorithm can understand.

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

### Tokenizer Demonstration

In [ ]:
example = dataset["train"][0]["text"]

tokens = tokenizer(example)

print(tokens)

### Applying Tokenization to the Full Dataset

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

### Inspecting the Processed Data

In [ ]:
tokenized_datasets

### Downsampling the Dataset for Faster Execution

In [ ]:
small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(300))
small_test = tokenized_datasets["test"].shuffle(seed=42).select(range(100))

### Dropping Redundant Features

In [ ]:
small_train = small_train.remove_columns(["text"])
small_test = small_test.remove_columns(["text"])

### Configuring Tensor Format for PyTorch

In [ ]:
small_train.set_format("torch")
small_test.set_format("torch")

### 3. Loading the Base BERT Architecture

Here, we import the foundational BERT model and configure its classification head to output exactly 4 categories.

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4,
    local_files_only=False
)

### 4. Setting Up Training Parameters

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1
)

### 5. Initializing the Model Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
)

### 6. Executing Model Fine-Tuning
Using the Hugging Face Trainer API, we train the model on our prepared dataset. This step adjusts the BERT weights to specifically recognize news context and vocabulary.

In [ ]:
trainer.train()

### 7. Performance Evaluation
To gauge how well our model generalizes to unseen data, we calculate two key metrics:
* **Accuracy Score**
* **Weighted F1-Score**

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

predictions = trainer.predict(small_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

accuracy = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average="weighted")

preds

In [ ]:
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

### 8. Exporting the Trained Model

In [ ]:
model.save_pretrained("news_bert_model")
tokenizer.save_pretrained("news_bert_model")

### 9. Interactive Web Application (Gradio)
To showcase the model's capability, we construct a user-friendly Gradio web app. This allows anyone to type a custom headline and instantly get an AI-driven topic prediction.

In [ ]:
import torch
import gradio as gr

# Label Mapping
labels = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

# Build Gradio Function
def predict_news(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    predicted_class = torch.argmax(outputs.logits).item()
    return labels[predicted_class]

# Build Gradio Interface & Launch
interface = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=2, placeholder="Enter a news headline..."),
    outputs="text",
    title="News Topic Classifier (BERT)",
    description="Enter a news headline and the model will predict its category."
)

interface.launch(share=True)

### 10. Final Thoughts & Conclusion
This project successfully illustrates the power of transfer learning in NLP. By taking a foundational BERT model and fine-tuning it on the AG News dataset, we built a highly capable classifier for short text.

We handled text tokenization, model training, and rigorous evaluation using Accuracy and F1 metrics across four news domains. The addition of a Gradio interface bridges the gap between backend AI training and front-end user experience, proving the practical value of transformer models in automating text categorization tasks.